In [3]:
import pandas as pd
import numpy as np


In [7]:
df = pd.read_csv('~/Downloads/archive (6)/winemag-data_first150k.csv')
df2 = pd.read_csv('~/Downloads/archive (6)/winemag-data-130k-v2.csv')
df3 = pd.read_json('~/Downloads/archive (6)/winemag-data-130k-v2.json')


In [8]:
from itertools import combinations

datasets = {
    "150k CSV": df,
    "130k CSV": df2,
    "JSON": df3
}

for name, data in datasets.items():
    print(f"{name}: {data.shape[0]:,} rows, {data.shape[1]} columns")
    print(f"Columns: {data.columns.tolist()}\n")

assert all("description" in data.columns for data in datasets.values())

review_sets = {}

for name, data in datasets.items():
    descriptions = data["description"].dropna().astype(str).str.strip().str.casefold()
    descriptions = descriptions[descriptions.ne("")]

    review_sets[name] = set(descriptions)

    print(f"{name}")
    print(f"  Unique descriptions: {descriptions.nunique():,}")
    print(f"  Duplicate descriptions: {descriptions.duplicated().sum():,}\n")

print("OVERLAP BETWEEN DATASETS")

for a, b in combinations(datasets, 2):
    overlap = len(review_sets[a] & review_sets[b])
    print(f"{a} vs {b}: {overlap:,} shared descriptions")

print("\nCOMBINED DATASET")

combined = pd.concat(
    [df, df3, df2],
    ignore_index=True
)

combined = combined.loc[:, ~combined.columns.str.startswith("Unnamed:")].copy()

combined["description_key"] = (
    combined["description"].astype("string").str.strip().str.casefold()
)

combined = combined.dropna(subset=["description_key"])
combined = combined[combined["description_key"].ne("")]

conflicts = (
    combined.groupby("description_key")["variety"]
    .nunique(dropna=True)
    .gt(1)
    .sum()
)

combined_clean = (
    combined.drop_duplicates(subset="description_key", keep="last")
    .drop(columns="description_key")
    .reset_index(drop=True)
)

print(f"Total reviews before deduplication: {len(combined):,}")
print(f"Unique reviews after deduplication: {len(combined_clean):,}")
print(f"Duplicates removed: {len(combined) - len(combined_clean):,}")
print(f"Descriptions with conflicting varieties: {conflicts:,}")

150k CSV: 150,930 rows, 11 columns
Columns: ['Unnamed: 0', 'country', 'description', 'designation', 'points', 'price', 'province', 'region_1', 'region_2', 'variety', 'winery']

130k CSV: 129,971 rows, 14 columns
Columns: ['Unnamed: 0', 'country', 'description', 'designation', 'points', 'price', 'province', 'region_1', 'region_2', 'taster_name', 'taster_twitter_handle', 'title', 'variety', 'winery']

JSON: 129,971 rows, 13 columns
Columns: ['points', 'title', 'description', 'taster_name', 'taster_twitter_handle', 'price', 'designation', 'variety', 'region_1', 'region_2', 'province', 'country', 'winery']

150k CSV
  Unique descriptions: 97,821
  Duplicate descriptions: 53,109

130k CSV
  Unique descriptions: 119,955
  Duplicate descriptions: 10,016

JSON
  Unique descriptions: 119,955
  Duplicate descriptions: 10,016

OVERLAP BETWEEN DATASETS
150k CSV vs 130k CSV: 48,346 shared descriptions
150k CSV vs JSON: 48,346 shared descriptions
130k CSV vs JSON: 119,955 shared descriptions

COMBIN

In [12]:
for name, data in {"150k CSV": df, "130k CSV": df2}.items():
    temp = data.copy()

    temp["description_key"] = (
        temp["description"].astype("string").str.strip().str.casefold()
    )

    duplicates = temp[
        temp["description_key"].notna()
        & temp["description_key"].ne("")
        & temp.duplicated("description_key", keep=False)
    ]

    examples = duplicates["description_key"].drop_duplicates().head(3)

    print(f"\n{'=' * 25} {name} {'=' * 25}")
    print(f"Duplicate descriptions: {duplicates['description_key'].nunique():,}")

    for i, desc in enumerate(examples, 1):
        print(f"\nEXAMPLE {i}")

        matching = duplicates[
            duplicates["description_key"] == desc
        ].drop(columns="description_key").reset_index(drop=True)

        with pd.option_context(
            "display.max_columns", None,
            "display.max_colwidth", None,
            "display.max_rows", None,
            "display.width", 1000
        ):
            display(matching.T)


========================= 150k CSV =========================
Duplicate descriptions: 39,284

EXAMPLE 1


,0,1
Unnamed: 0,25,305
country,New Zealand,New Zealand
description,"Yields were down in 2015, but intensity is up, giving this medium-bodied, silky wine the potential to drink well through at least 2025. Hickory smoke outlines white peach before ending in a long flurry of lime zest.","Yields were down in 2015, but intensity is up, giving this medium-bodied, silky wine the potential to drink well through at least 2025. Hickory smoke outlines white peach before ending in a long flurry of lime zest."
designation,Maté's Vineyard,Maté's Vineyard
points,94,94
price,57.0,57.0
province,Kumeu,Kumeu
region_1,NaN,NaN
region_2,NaN,NaN
variety,Chardonnay,Chardonnay



EXAMPLE 2


,0,1
Unnamed: 0,29,300
country,US,US
description,"This standout Rocks District wine brings earth shaking aromas of black-olive brine, tapenade, green olive, stargazer lilies, orange peel and crushed gravel. The smoked meat, charcuterie and blue-fruit flavors don't hold back, bringing a lovely sense of texture and detail. It's an intense wine that completely demands your attention.","This standout Rocks District wine brings earth shaking aromas of black-olive brine, tapenade, green olive, stargazer lilies, orange peel and crushed gravel. The smoked meat, charcuterie and blue-fruit flavors don't hold back, bringing a lovely sense of texture and detail. It's an intense wine that completely demands your attention."
designation,The Funk Estate,The Funk Estate
points,94,94
price,60.0,60.0
province,Washington,Washington
region_1,Walla Walla Valley (WA),Walla Walla Valley (WA)
region_2,Columbia Valley,Columbia Valley
variety,Syrah,Syrah



EXAMPLE 3


...

,0,1
Unnamed: 0,9,102758
country,France,France
description,This has great depth of flavor with its fresh apple and pear fruits and touch of spice. It's off dry while balanced with acidity and a crisp texture. Drink now.,This has great depth of flavor with its fresh apple and pear fruits and touch of spice. It's off dry while balanced with acidity and a crisp texture. Drink now.
designation,Les Natures,Les Natures
points,87,87
price,27.0,27.0
province,Alsace,Alsace
region_1,Alsace,Alsace
region_2,NaN,NaN
taster_name,Roger Voss,Roger Voss



EXAMPLE 2


,0,1
Unnamed: 0,10,102759
country,US,US
description,"Soft, supple plum envelopes an oaky structure in this Cabernet, supported by 15% Merlot. Coffee and chocolate complete the picture, finishing strong at the end, resulting in a value-priced wine of attractive flavor and immediate accessibility.","Soft, supple plum envelopes an oaky structure in this Cabernet, supported by 15% Merlot. Coffee and chocolate complete the picture, finishing strong at the end, resulting in a value-priced wine of attractive flavor and immediate accessibility."
designation,Mountain Cuvée,Mountain Cuvée
points,87,87
price,19.0,19.0
province,California,California
region_1,Napa Valley,Napa Valley
region_2,Napa,Napa
taster_name,Virginie Boone,Virginie Boone



EXAMPLE 3


,0,1
Unnamed: 0,11,102760
country,France,France
description,"This is a dry wine, very spicy, with a tight, taut texture and strongly mineral character layered with citrus as well as pepper. It's a food wine with its almost crisp aftertaste.","This is a dry wine, very spicy, with a tight, taut texture and strongly mineral character layered with citrus as well as pepper. It's a food wine with its almost crisp aftertaste."
designation,NaN,NaN
points,87,87
price,30.0,30.0
province,Alsace,Alsace
region_1,Alsace,Alsace
region_2,NaN,NaN
taster_name,Roger Voss,Roger Voss


In [13]:
a = df.copy()
b = df2.copy()

for data in [a, b]:
    data["description_key"] = (
        data["description"].astype("string").str.strip().str.casefold()
    )

shared = (
    a.loc[
        a["description_key"].notna()
        & a["description_key"].ne("")
        & a["description_key"].isin(b["description_key"]),
        "description_key"
    ]
    .drop_duplicates()
    .head(2)
)

with pd.option_context(
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.max_rows", None,
    "display.width", 1000
):
    for i, desc in enumerate(shared, 1):
        old = a[a["description_key"] == desc].iloc[[0]]
        new = b[b["description_key"] == desc].iloc[[0]]

        comparison = pd.concat([
            old.drop(columns=["description_key", "Unnamed: 0"], errors="ignore").T,
            new.drop(columns=["description_key", "Unnamed: 0"], errors="ignore").T
        ], axis=1)

        comparison.columns = ["150k CSV", "130k CSV"]

        print(f"\n{'=' * 20} EXAMPLE {i} {'=' * 20}")
        display(comparison)


==================== EXAMPLE 1 ====================


,150k CSV,130k CSV
country,US,US
description,"Bergström has made a Shea designate since 2003, intent on showcasing a “pretty” style of the vineyard. Here are lovely aromatics, with grape jelly, rose petals and plum in the nose, and a complex run of red fruits in the mouth. Streaks of cola, brown sugar and more come up in a generous finish.","Bergström has made a Shea designate since 2003, intent on showcasing a “pretty” style of the vineyard. Here are lovely aromatics, with grape jelly, rose petals and plum in the nose, and a complex run of red fruits in the mouth. Streaks of cola, brown sugar and more come up in a generous finish."
designation,Shea Vineyard,Shea Vineyard
points,94,94
price,62.0,62.0
province,Oregon,Oregon
region_1,Willamette Valley,Willamette Valley
region_2,NaN,NaN
variety,Pinot Noir,Pinot Noir
winery,Bergström,Bergström



==================== EXAMPLE 2 ====================


,150k CSV,130k CSV
country,US,US
description,"Focused and dense, this intense wine captures the essence of ripe raspberries. Aged in 50% new French oak, it's got a strong vein of coffee and mocha. The flavors broaden out and penetrate, finishing with highlights of cedar, cinnamon and dark chocolate. Drink now through the 2020s, but decant if you want the full expression.","Focused and dense, this intense wine captures the essence of ripe raspberries. Aged in 50% new French oak, it's got a strong vein of coffee and mocha. The flavors broaden out and penetrate, finishing with highlights of cedar, cinnamon and dark chocolate. Drink now through the 2020s, but decant if you want the full expression."
designation,Abetina,Abetina
points,94,94
price,105.0,105.0
province,Oregon,Oregon
region_1,Willamette Valley,Willamette Valley
region_2,Willamette Valley,Willamette Valley
variety,Pinot Noir,Pinot Noir
winery,Ponzi,Ponzi


# Merging datasets
Account for dupes within and between datasets

In [14]:
combined = pd.concat([df, df2], ignore_index=True)

combined = combined.drop(columns="Unnamed: 0", errors="ignore")
combined = combined.dropna(subset=["description", "variety"]).copy()

combined["description_key"] = (
    combined["description"].str.strip().str.casefold()
)

combined = combined[combined["description_key"].ne("")]

conflicts = (
    combined.groupby("description_key")["variety"]
    .nunique()
)

conflicting_keys = conflicts[conflicts > 1].index

combined_clean = (
    combined[~combined["description_key"].isin(conflicting_keys)]
    .drop_duplicates(subset="description_key", keep="last")
    .drop(columns="description_key")
    .reset_index(drop=True)
)

print("Original rows:", len(df) + len(df2))
print("Final rows:", len(combined_clean))
print("Conflicting descriptions excluded:", len(conflicting_keys))
print("Final shape:", combined_clean.shape)

combined_clean.head()

Original rows: 280901
Final rows: 169407
Conflicting descriptions excluded: 22
Final shape: (169407, 13)


,country,description,designation,points,price,province,region_1,region_2,variety,winery,taster_name,taster_twitter_handle,title
0,US,This tremendous 100% varietal wine hails from ...,Martha's Vineyard,96,235.0,California,Napa Valley,Napa,Cabernet Sauvignon,Heitz,NaN,NaN,NaN
1,Spain,"Ripe aromas of fig, blackberry and cassis are ...",Carodorum Selección Especial Reserva,96,110.0,Northern Spain,Toro,NaN,Tinta de Toro,Bodega Carmen Rodríguez,NaN,NaN,NaN
2,US,Mac Watson honors the memory of a wine once ma...,Special Selected Late Harvest,96,90.0,California,Knights Valley,Sonoma,Sauvignon Blanc,Macauley,NaN,NaN,NaN
3,US,"This spent 20 months in 30% new French oak, an...",Reserve,96,65.0,Oregon,Willamette Valley,Willamette Valley,Pinot Noir,Ponzi,NaN,NaN,NaN
4,France,"This is the top wine from La Bégude, named aft...",La Brûlade,95,66.0,Provence,Bandol,NaN,Provence red blend,Domaine de la Bégude,NaN,NaN,NaN


In [15]:
combined_clean.to_parquet("df_combined.parquet", index=False)